# European Flight Delay Prediction — AI Training & Evaluation

This notebook will train and compare machine learning models to predict whether a European flight will be delayed by more than 15 minutes.

It will use flight data generated by '03_data_splitting.ipynb', validates records separating rejected records and validateed in '06_data_pipline.ipynb'
- Training: 80,000 records
- Development: 10,000 records
- Test: 10,000 records - this will be unused during model selection and hyperparameter optimisation

The following models will be evaluated
- Simple baseline classifier
- Support Vector Machine (SVM) using scikit-learn
- Multi-Layer Perceptron (MLP) using PyTorch

Evaluation metrics:
- Accuracy, precision, recall and F1-score
- Confusion matrix
- Training time

MLflow will track experiments, hyperparameters, metrics and model
artifacts to support reproducibility.

The best model will be selected using development-set performance
before final evaluation on the held-out test dataset.


**Please use T4 GPU Runtime when running this notebook in Google Colab**

## Prerequisites

The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment before installing the required dependencies.

In [ ]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"

%cd /content/
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR} will pull latest version")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")
    # Change the current working directory to the project root
    %cd {REPO_DIR}

## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [ ]:
%pip install -r requirements.txt

## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

Review the folllowing values before running to ensure they are correct for your run:

*   PROJECT_ID
*   BUCKET_NAME
*   SPLIT_RUN_ID - this should reference existing notebook 03 split run


In [ ]:

from google.colab import auth
auth.authenticate_user()

from pathlib import Path
from google.cloud import storage
import pandas as pd
import json

PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"

# Existing Notebook 03 split run
SPLIT_RUN_ID = "20261020T065554Z-cc2d29384141"
SPLIT_PREFIX = f"processed/splits/runs/{SPLIT_RUN_ID}"

TARGET_COLUMN = "delayed_15min"

WORK_DIR = Path("/content/flight_preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TRAIN = WORK_DIR / "flights_train.parquet"
LOCAL_DEV = WORK_DIR / "flights_dev.parquet"
LOCAL_TEST = WORK_DIR / "flights_test.parquet"
LOCAL_SAMPLE = WORK_DIR / "flights_sample_100.parquet"


client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

# Confirm the source run exists
provenance_blob = bucket.blob(
    f"{SPLIT_PREFIX}/provenance.json"
)

if not provenance_blob.exists():
    raise FileNotFoundError(
        f"Split run not found: gs://{BUCKET_NAME}/{SPLIT_PREFIX}"
    )

source_provenance = json.loads(
    provenance_blob.download_as_text()
)

assert source_provenance["run_id"] == SPLIT_RUN_ID

print("Connected to Google Cloud Storage.")
print("Source split run:", SPLIT_RUN_ID)
print("Source split sizes:", source_provenance["split_rows"])


## 3. Add additional imports and reproducibility settings

In [ ]:

import random
import time

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import torch
import mlflow

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("MLflow:", mlflow.__version__)
print("Matplotlib:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())


## 4. Load and Verify Training Datasets

Load the datasets created in Notebook 3 and validated in Notebook 6 and verify the following before training
- Number of records
- Available feature columns
- Distribution of target variable delayed_15min
- Datasets contain the expected 80/10/10 splits

Note: The test dataset will only be inspected for structural integrity, its labels and performance will not be used for model selection or hyperparameter optimisation

In [ ]:

from google.cloud import storage

split_files = {
    "flights_train": LOCAL_TRAIN,
    "flights_dev": LOCAL_DEV,
    "flights_test": LOCAL_TEST,
}

for split_name, local_path in split_files.items():
    blob_path = f"{SPLIT_PREFIX}/{split_name}.parquet"

    print(f"Downloading {split_name}: {blob_path}")

    bucket.blob(blob_path).download_to_filename(
        str(local_path)
    )

train_df = pd.read_parquet(LOCAL_TRAIN)
dev_df = pd.read_parquet(LOCAL_DEV)
test_df = pd.read_parquet(LOCAL_TEST)

print("\nDataset sizes:")
print("Training:", len(train_df))
print("Development:", len(dev_df))
print("Test:", len(test_df))

assert len(train_df) == 80000
assert len(dev_df) == 10000
assert len(test_df) == 10000

print("\nTraining dataset columns:")
print(train_df.columns.tolist())

print("\nTraining target distribution:")
print(train_df["delayed_15min"].value_counts())
print(train_df["delayed_15min"].value_counts(normalize=True).round(3))

print("\nDataset loading and size checks passed.")


## 5. Feature Selection and Majority-Class Baseline Model

The target variable is delayed_15min which indicates if a flight was delayed by more than 15 minutes.

The training set is imbalanced with approximately 8.4% flights classified as delayed so a majority class baseline will be established before training more complex models.

Features selected will be based on information that is likely to be available before flight outcome is known avoiding data leakage.

The primary model selection metric will be F1-score for the delayed class which combines precision (of all flights predicted would be delayed how many were) and recall (of all the flights actually delayed how many did the model identify)

The F1-score will also be supported by precision, recall, accuracy and average precision (how good it is at identifying highest-risk flights across different cut offs)

In [ ]:
# Feature selection

from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

CATEGORICAL_FEATURES = [
    "adep",
    "ades",
    "route",
    "typecode",
    "model",
    "icao_aircraft_class",
    "icao_operator",
]

NUMERIC_FEATURES = [
    "source_year",
    "month",
    "expected_duration_min",
    "day_of_week",
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = "delayed_15min"

def prepare_features(df):
    data = df.copy()

    # Extract calendar month from YYYYMM
    data["month"] = (
        pd.to_numeric(data["source_month"], errors="coerce")
        % 100
    )

    # Derive weekday: Monday=0, Sunday=6
    data["day_of_week"] = pd.to_datetime(
        data["dof"], errors="coerce"
    ).dt.dayofweek

    return data[FEATURES]

# input features
X_train = prepare_features(train_df)
X_dev = prepare_features(dev_df)

# target
y_train = train_df[TARGET].astype(int)
y_dev = dev_df[TARGET].astype(int)

print("Training features:", X_train.shape)
print("Development features:", X_dev.shape)
print("\nMissing values in training features:")
print(X_train.isna().sum())

assert X_train.shape == (80000, len(FEATURES))
assert X_dev.shape == (10000, len(FEATURES))

print("\nFeature preparation completed.")


In [ ]:
# Baseline Model - always predict the most common outcome i.e. 0 flight not delayed in our case

baseline = DummyClassifier(strategy="most_frequent") # Use scikit-learn dummy classfifier predicting whichever class occurs most often in the training set
baseline.fit(X_train, y_train) # train the model using x_train input features where y_train contains the target values

# use the trained model to predict if the dev flights are delayed, will predict 0 for all
baseline_predictions = baseline.predict(X_dev)

# same but predict probabilities rather than just 0/1, will produce two coluns probability of 0 and 1, and we take the second column i.e. probability of 1
baseline_probabilities = baseline.predict_proba(X_dev)[:, 1]

# Create json object with results
baseline_results = {
    "accuracy": accuracy_score(y_dev, baseline_predictions),  # proportion of predictions that were correct
    "precision": precision_score(                             # of those it predicted would be delayed how many actually were
        y_dev, baseline_predictions, zero_division=0          # zero_division return 0 rather than undefined mertric
    ),
    "recall": recall_score(                                   # of those actually delayed how many did the model identify
        y_dev, baseline_predictions, zero_division=0
    ),
    "f1": f1_score(                                           # combines precision and recall
        y_dev, baseline_predictions, zero_division=0
    ),
    "average_precision": average_precision_score(             # how well model ranks geniunely delayed flights above non delayed flights using prediction scores
        y_dev, baseline_probabilities
    ),
}

print("Majority-class baseline results:")
for metric, value in baseline_results.items():
    print(f"{metric}: {value:.4f}")

print("\nDevelopment target distribution:")
print(y_dev.value_counts(normalize=True).round(4))


## 6. Data Preprocessing for Model Training

Convert the features into format suitable for machine learning - fitted on training data only and applied to development data without refitting to avoid data leakage and ensure consistent transformations

Categorical Features
- Replace missing values with "Unknown"
- Convert categories into numerical values using one-hot encoding

Numerical Features
- Replace missing values with the training median
- Standardize values using Standard Scalar

In [ ]:

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"                # populate missing values with Unknown
    )),
    ("encoder", OneHotEncoder(
        # Create a column for each category found during training.
        # Use 1 if it applies, otherwise 0.
        handle_unknown="ignore" # allow new unseen categories without errors
    ))
])

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median"                   # populate missing values with median
    )),
    ("scaler", StandardScaler())            # standardize values
])

# Combine categorical and numerical transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
        ("num", numeric_transformer, NUMERIC_FEATURES)
    ]
)

# Fit i.e. learn preprocessing rules using training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply same preprocessing to development data without learning anything else/refitting
X_dev_processed = preprocessor.transform(X_dev)

print("Training shape:", X_train_processed.shape)
print("Development shape:", X_dev_processed.shape)

# this should product csr_matrix i.e. Compressed Sparse Row as most values in the one-hot encoded data will be zeros, it will only
# store the non zero values and information about their position to reduce memory usage
print("Output type:", type(X_train_processed).__name__)
print("Number of processed features:", X_train_processed.shape[1])

assert X_train_processed.shape[0] == 80000
assert X_dev_processed.shape[0] == 10000
assert X_train_processed.shape[1] == X_dev_processed.shape[1]

print("\nPreprocessing completed successfully.")


In [ ]:
# Preprocessing validation
from scipy import sparse

def check_processed_data(name, X):
    values = X.data if sparse.issparse(X) else np.asarray(X)

    missing_count = np.isnan(values).sum()
    infinite_count = np.isinf(values).sum()

    print(f"{name}:")
    print(f"  Missing values: {missing_count}")
    print(f"  Infinite values: {infinite_count}")

    assert missing_count == 0
    assert infinite_count == 0

check_processed_data("Training", X_train_processed)
check_processed_data("Development", X_dev_processed)

print("\nAll preprocessing validation checks passed.")


## 7. Initial Model Training — Logistic Regression

Train a Logistic Regression classifier as first predictive model. It is a model that is suitable for binary classification and supports high dimensional sparce feature matrices (csr_matrix).

It will be trained using the 80,000 training records and evaluated on the 10,000 development records.

Results will be compared against the majority-class baseline model using accuracy, precision, recall, F1-score and average precision.

This will provide an initial preditive benchmark before training and optomizing SVM and PyTorch neural network models

In [ ]:

from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model
logistic_model = LogisticRegression(
    max_iter=1000,   # allow up to 1000 iterations rather than default 200
    random_state=SEED
)

# Train the model and measure training time
start_time = time.perf_counter()

# Learn relationships between features and delays
logistic_model.fit(X_train_processed, y_train)

training_time = time.perf_counter() - start_time

# Predict on the development dataset - produces classification of 0/1
logistic_predictions = logistic_model.predict(X_dev_processed)

# Predict probabilitiy of 0/1
logistic_probabilities = logistic_model.predict_proba(
    X_dev_processed
)[:, 1]

# Calculate evaluation metrics
logistic_results = {
    "accuracy": accuracy_score(y_dev, logistic_predictions),
    "precision": precision_score(
        y_dev, logistic_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_dev, logistic_predictions, zero_division=0
    ),
    "f1": f1_score(
        y_dev, logistic_predictions, zero_division=0
    ),
    "average_precision": average_precision_score(
        y_dev, logistic_probabilities
    ),
    "training_time_seconds": training_time
}

print("Logistic Regression results:")
for metric, value in logistic_results.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_dev,
    logistic_predictions,
    zero_division=0
))


In [ ]:
# Compare with majority-class baseline

In [ ]:

comparison = pd.DataFrame({
    "Baseline": baseline_results,
    "Logistic Regression": logistic_results
})

print("Model comparison:")
display(comparison.round(4))


In [ ]:
# Calculate Confusion Matrix i.e. how many it classified correctly and wrongly
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_dev, logistic_predictions)

print("Logistic Regression confusion matrix:")
print(cm)

# Display the confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Delayed", "Delayed"]
).plot(cmap="Blues", values_format="d")

plt.title("Logistic Regression - Development Dataset")
plt.show()
